In [ ]:
import os
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
    check=True,
)

from huggingface_hub import hf_hub_download

# Ang bucket ni boss. Private ito, kaya kailangan ng HF_TOKEN — pero may
# public mirror na eksaktong pareho ang file, para makapag-test kahit wala.
PRIVATE_REPO = "Rexmay/SeaLLM-7B-v2-gguf-bucket"
PUBLIC_REPO = "SeaLLMs/SeaLLM-7B-v2-gguf"
FILES = [
    "SeaLLM-7B-v2.q4_0.gguf",   # ang model (4-bit)
    "seallm.preset.json",        # prompt format
]
LOCAL_DIR = os.environ.get("SEALLM_DIR", "./seallm")
os.makedirs(LOCAL_DIR, exist_ok=True)

GGUF = os.path.join(LOCAL_DIR, "SeaLLM-7B-v2.q4_0.gguf")
MIN_BYTES = 3 * 1024 ** 3


def _read_env_file(path="./.env"):
    values = {}
    if not os.path.exists(path):
        return values
    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            values[key.strip()] = val.strip().strip('"').strip("'")
    return values


def _complete(path):
    """A GGUF starts with the magic bytes; a truncated download does not end well."""
    if not os.path.exists(path) or os.path.getsize(path) < MIN_BYTES:
        return False
    with open(path, "rb") as fh:
        return fh.read(4) == b"GGUF"


HF_TOKEN = os.environ.get("HF_TOKEN", "").strip() or _read_env_file().get("HF_TOKEN", "").strip()
REPO_ID = os.environ.get("SEALLM_REPO_ID", "").strip()

if _complete(GGUF):
    print(f"✅ Nandiyan na ang model ({os.path.getsize(GGUF) / 1024 ** 3:.2f} GB). "
          "Laktawan ang download.")
else:
    # Walang token = subukan ang public mirror. May token = gamitin ang bucket ni boss.
    if not REPO_ID:
        REPO_ID = PRIVATE_REPO if HF_TOKEN else PUBLIC_REPO
    if REPO_ID == PUBLIC_REPO and not HF_TOKEN:
        print(f"ℹ️ Walang HF_TOKEN, kaya public mirror ang gagamitin: {PUBLIC_REPO}")
        print("   Pareho ito ng file sa bucket ni boss. Para sa private bucket, "
              "lagyan ng HF_TOKEN ang .env.")

    for fname in FILES:
        try:
            path = hf_hub_download(
                repo_id=REPO_ID,
                filename=fname,
                local_dir=LOCAL_DIR,
                token=HF_TOKEN or None,
            )
        except Exception as exc:
            raise SystemExit(
                f"Hindi ma-download ang {fname} mula sa {REPO_ID}.\n"
                f"Dahilan: {exc}\n"
                "Kung 401/403 ito, private ang bucket — lagyan ng HF_TOKEN ang .env "
                "at tiyaking may access ang account mo.\n"
                "Walang token? Patakbuhin ulit nang walang HF_TOKEN para gamitin "
                f"ang public mirror ({PUBLIC_REPO})."
            ) from exc
        print("Na-download:", path)

if not _complete(GGUF):
    raise SystemExit(
        f"Sira o kulang ang {GGUF} ({os.path.getsize(GGUF) / 1024 ** 3:.2f} GB). "
        "Burahin ito at patakbuhin ulit ang cell."
    )
print(f"✅ Model OK: {os.path.getsize(GGUF) / 1024 ** 3:.2f} GB, valid GGUF.")


In [2]:
# CELL 1 — Install
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "pydantic", "requests"],
    check=True,
)

print("⏳ Ini-install ang llama-cpp-python (maaaring abutin ng ilang minuto)...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--prefer-binary",
     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu",
     "llama-cpp-python"],
)
assert r.returncode == 0, "Hindi na-install ang llama-cpp-python. Subukang i-run ulit ang cell."
print("✅ Cell 1 done.")

⏳ Ini-install ang llama-cpp-python (maaaring abutin ng ilang minuto)...
✅ Cell 1 done.


In [3]:
# ============================================================
# CELL 2 — Core: Math + Morals + Inventory + Live Selling
# ============================================================
import os
import re
import json
import random
from pathlib import Path
from datetime import datetime

DATA_FILE = Path("./tindahan_data.json")


# ==========================================================
# 🧮 MATH LOGIC
# ==========================================================
class MathLogic:
    VAT_RATE = 0.12
    ROUND_PLACES = 2
    MAX_SAFE_AMOUNT = 10_000_000

    @staticmethod
    def _validate_amount(x, name="amount"):
        if x is None:
            raise ValueError(f"{name} ay walang value.")
        if not isinstance(x, (int, float)):
            raise ValueError(f"{name} ay dapat numero.")
        if x < 0:
            raise ValueError(f"{name} ay hindi pwedeng negatibo.")
        if x > MathLogic.MAX_SAFE_AMOUNT:
            raise ValueError(f"{name} ay masyadong malaki.")
        return float(x)

    @staticmethod
    def round_money(x):
        return round(float(x) + 1e-9, MathLogic.ROUND_PLACES)

    @staticmethod
    def compute_total(price, qty):
        price = MathLogic._validate_amount(price, "presyo")
        if not isinstance(qty, (int, float)) or qty < 0:
            raise ValueError("Dami ay dapat non-negative na numero.")
        if qty == 0:
            return 0.0, {"price": price, "qty": qty, "subtotal": 0.0}
        subtotal = price * qty
        return MathLogic.round_money(subtotal), {"price": price, "qty": qty, "subtotal": subtotal}

    @staticmethod
    def compute_discount(subtotal, percent=0, fixed=0):
        subtotal = MathLogic._validate_amount(subtotal, "subtotal")
        percent = MathLogic._validate_amount(percent, "percent")
        fixed = MathLogic._validate_amount(fixed, "fixed")
        if percent > 100:
            raise ValueError("Discount percent ay hindi pwedeng lumampas sa 100%.")
        pct_amt = subtotal * (percent / 100.0)
        total_discount = min(pct_amt + fixed, subtotal)
        final = subtotal - total_discount
        return MathLogic.round_money(final), {
            "subtotal": subtotal,
            "percent_amt": MathLogic.round_money(pct_amt),
            "fixed_amt": fixed,
            "total_discount": MathLogic.round_money(total_discount),
            "final": MathLogic.round_money(final),
        }

    @staticmethod
    def compute_change(cash_paid, total):
        cash_paid = MathLogic._validate_amount(cash_paid, "bayad")
        total = MathLogic._validate_amount(total, "total")
        if cash_paid < total:
            raise ValueError(f"Kulang ang bayad: ₱{cash_paid:.2f} < ₱{total:.2f}.")
        return MathLogic.round_money(cash_paid - total)

    @staticmethod
    def compute_vat(gross_amount, inclusive=True):
        gross = MathLogic._validate_amount(gross_amount, "amount")
        if inclusive:
            net = gross / (1 + MathLogic.VAT_RATE)
            vat = gross - net
        else:
            vat = gross * MathLogic.VAT_RATE
            net = gross
        return {
            "net": MathLogic.round_money(net),
            "vat": MathLogic.round_money(vat),
            "gross": MathLogic.round_money(gross),
            "rate": MathLogic.VAT_RATE,
            "inclusive": inclusive,
        }

    @staticmethod
    def compute_profit_margin(sell_price, buy_price):
        sell_price = MathLogic._validate_amount(sell_price, "sell")
        buy_price = MathLogic._validate_amount(buy_price, "buy")
        if sell_price == 0:
            return {"margin_php": 0, "margin_pct": 0, "markup_pct": 0}
        margin = sell_price - buy_price
        margin_pct = (margin / sell_price * 100) if sell_price else 0
        markup_pct = (margin / buy_price * 100) if buy_price else 0
        return {
            "margin_php": MathLogic.round_money(margin),
            "margin_pct": round(margin_pct, 2),
            "markup_pct": round(markup_pct, 2),
        }

    @staticmethod
    def compute_roi(net_profit, capital):
        if capital <= 0:
            return 0.0
        return round((net_profit / capital) * 100, 2)

    @staticmethod
    def compute_break_even(monthly_expenses, avg_margin_pct):
        if avg_margin_pct <= 0:
            return None
        return MathLogic.round_money(monthly_expenses / (avg_margin_pct / 100.0))

    @staticmethod
    def parse_money(text):
        if not text:
            return None
        t = text.replace(",", "").replace("₱", "").replace("php", "").strip()
        m = re.search(r'(\d+(?:\.\d+)?)\s*([kKmM])?', t)
        if not m:
            return None
        val = float(m.group(1))
        suf = (m.group(2) or "").lower()
        if suf == "k":
            val *= 1_000
        elif suf == "m":
            val *= 1_000_000
        return MathLogic.round_money(val)


# ==========================================================
# 💛 MORAL CONDUCT
# ==========================================================
MORAL_CONDUCT = {
    "core_values": {
        "katapatan": "Laging magsabi ng totoo sa presyo, stock, kalidad, at oras ng delivery.",
        "paggalang": "Maging magalang sa lahat — buyer, seller, at kapwa.",
        "katarungan": "Patas sa lahat. Walang fake scarcity, fake reviews, o panlilinlang.",
        "responsibilidad": "Sumunod sa batas (BIR, DTI, RA 10173).",
        "malasakit": "Tulungan ang buyer na mahanap ang tamang produkto.",
        "kapanagutan": "Aminin kapag hindi alam. Huwag magsinungaling.",
        "integridad": "Hindi magtataas ng presyo nang palihim, hindi magtatago ng depekto.",
    },
    "rules_of_conduct": [
        "HUWAG magsinungaling tungkol sa stock, presyo, quality, o delivery time.",
        "HUWAG gumawa ng fake review, fake testimonial, o fake buyer message.",
        "HUWAG gumamit ng fake scarcity ('3 na lang!' kung marami pa).",
        "HUWAG mag-claim ng health/beauty benefits na walang FDA approval.",
        "HUWAG magbenta ng bawal.",
        "HUWAG mang-harass, mang-insulto, o mang-discriminate.",
        "HUWAG ibunyag ang personal na info ng buyer.",
        "HUWAG mangako ng hindi kayang tuparin.",
        "HUWAG mag-udyok ng impulse buying ('bili na o maiiyak ka').",
        "HUWAG mag-imbento ng quote, batas, o FDA approval.",
        "DAPAT ipaalam: 'Ako ay AI assistant. Ang seller ang may final approval.'",
        "DAPAT i-handle ang reklamo nang mahinahon.",
        "DAPAT i-respeto ang desisyon ng buyer kung ayaw bumili.",
    ],
    "korte_kita_rules": [
        "Ibenta nang may malinaw na kita.",
        "Ipakita ang breakdown ng presyo kapag tinanong.",
        "Ipaliwanag ang shipping fee bago mag-order.",
        "Ipaalala ang return/refund policy bago magbayad.",
        "Kapag nalugi sa isang deal, aminin.",
    ],
}


# ==========================================================
# 🎯 BUSINESS PRINCIPLES
# ==========================================================
BUSINESS_PRINCIPLES = {
    "mission": (
        "Tulungan ang maliliit na tindero/tindera at live seller na magkaroon ng "
        "malinaw na record ng puhunan, benta, at kita — para lumago ang negosyo "
        "nang may tamang desisyon. Makipag-benta nang tapat at kapaki-pakinabang "
        "sa parehong seller at buyer."
    ),
    "vision": (
        "Isang Pilipinas kung saan bawat sari-sari store at live seller ay may "
        "access sa financial literacy at digital tools na abot-kaya at may integridad."
    ),
    "tasks": [
        "I-track ang bawat produkto: puhunan, presyo ng benta, at stock.",
        "Ipakita ang gross income, net income, ROI, break-even, at margin.",
        "Sumagot sa live chat (presyo, stock, shipping, COD).",
        "Ipakita ang totoong stock at totoong presyo.",
        "I-record ang bawat benta at i-update ang stock.",
        "Ipaalala ang legal na pagsunod (BIR, DTI, RA 10173).",
    ],
    "compliance": [
        "Bawal ang pekeng resibo at hindi pagbabayad ng buwis.",
        "Ipaalala ang business permit at BIR registration.",
        "Igalang ang consumer rights.",
        "Data privacy (RA 10173) — huwag i-share ang buyer info.",
        "Hindi ako accountant/abogado — magpakonsulta sa propesyonal.",
        "Bawal fake scarcity, fake reviews, at health claims na walang FDA.",
    ],
    "disclosure": (
        "Ako ay AI assistant ng seller na ito. Ang seller ang may final approval "
        "sa lahat ng promo at presyo."
    ),
}


# ==========================================================
# 📝 ONBOARDING
# ==========================================================
ONBOARDING_QUESTIONS = [
    {"key": "store_name", "type": "text",
     "question": "Boss, ano po ang pangalan ng tindahan/shop ninyo? (Hal. SariSwerte)"},
    {"key": "owner_name", "type": "text",
     "question": "Ano po ang pangalan ninyo, boss?"},
    {"key": "live_platform", "type": "text",
     "question": "Saan po kayo nag-live selling? (FB Live, TikTok, Shopee, o 'walk-in')"},
    {"key": "capital", "type": "number",
     "question": "Magkano po ang puhunan ninyo? (Hal. 5000)"},
    {"key": "monthly_expenses", "type": "number",
     "question": "Magkano po ang buwanang gastos ninyo? (Kuryente, tubig, internet, renta)"},
    {"key": "shipping_fee", "type": "number",
     "question": "Magkano po ang standard shipping fee? (0 kung pickup)"},
    {"key": "min_free_ship", "type": "number",
     "question": "Magkano po ang minimum order para sa free shipping? (0 kung wala)"},
    {"key": "return_policy", "type": "text",
     "question": "Ano po ang return/refund policy ninyo? (Hal. '7-day return kung may sira')"},
]

PRODUCT_INPUT_PROMPT = (
    "Boss, sabihin mo ang produkto nang ganito para madagdag ko:\n"
    "  ADD: pangalan | presyo_bili | presyo_benta | stock | unit\n"
    "  Halimbawa: ADD: Coke 1.5L | 65 | 75 | 20 | bote\n"
    "I-type ang 'DONE' kapag tapos na."
)


# ==========================================================
# 🎬 TAKE-OVER
# ==========================================================
TAKE_OVER_RULES = {
    "allowed": [
        "Seller: 'AI ikaw na' / 'take over' / 'AI sagutin mo'.",
        "30s walang sagot sa direktang tanong ng buyer.",
        "Paulit-ulit ang tanong at hindi nasasagot.",
        "Mabilisang math (presyo × qty, shipping, discount).",
    ],
    "forbidden": [
        "Personal na usapan (reklamo, refund dispute) — ipasa sa seller.",
        "Sensitibo (health claims, legal questions).",
        "Seller ay aktibong nagsasalita — hintayin ang cue.",
        "Pekeng customer message o fake review.",
        "Imbentong stock o presyo.",
    ],
    "hand_back_phrases": [
        "Ipaubaya ko na po sa seller — si {owner} na po ang sasagot.",
        "Kailangan po ng personal na sagot ng seller dito.",
        "Seller, kailangan po ng desisyon ninyo dito.",
    ],
}

TAKE_OVER_TRIGGERS = [
    r"\bAI[, ]+ikaw na\b", r"\btake over\b", r"\bAI[, ]+sagutin mo\b",
    r"\bikaw na muna\b", r"\bAI[, ]+ikaw na muna\b",
]
HAND_BACK_TRIGGERS = [r"\bako na\b", r"\bseller na\b", r"\bhand back\b", r"\bstop take over\b"]


# ==========================================================
# 🎙️ LIVE SCRIPTS
# ==========================================================
LIVE_SCRIPTS = {
    "opening": [
        "Hello mga ka-live! Welcome sa {store}! Ako po si {agent}, AI co-host ni {owner}. "
        "May bagong stock at legit na promo tayo — tara, tingnan natin!",
        "Live na live na po tayo! Ako si {agent}, kasama si {owner}. Comment 'ORDER' kung may gusto.",
    ],
    "greeting_buyer": [
        "Welcome {buyer}! Comment lang po kung may tanong.",
        "Salamat sa pag-join, {buyer}! Anong hanap ninyo?",
    ],
    "product_pitch": [
        "{product} po, mga boss — ₱{price} lang bawat {unit}. May stock pa kaming {stock}. Sulit 'to!",
        "Tingnan ninyo itong {product}! Presyong {price}, quality guaranteed.",
    ],
    "answer_price": [
        "₱{price} po ang {product}, boss. Kasama na packaging. COD dagdag ₱{ship_fee}.",
    ],
    "answer_stock": [
        "Meron pa po tayong {stock} {unit} ng {product}.",
        "Konti na lang po — {stock} na lang ang {product}. I-reserve na natin?",
    ],
    "answer_shipping": [
        "Shipping: ₱{ship_fee} via J&T/LBC, 2-4 days. Pwede COD.",
        "Free shipping kapag umabot ng ₱{min_amount}. Pwede rin meet-up.",
    ],
    "answer_payment": [
        "GCash, Maya, o COD. May resibo kami sa bawat order.",
    ],
    "closing": [
        "Salamat mga boss! Follow para sa next live namin!",
        "Maraming salamat! I-message namin kayo para sa confirmation. Ingat!",
    ],
    "sold_out": [
        "Sold out na po ang {product}! May iba pa tayong stock — tingnan natin.",
    ],
    "low_stock_honest": [
        "Sa totoo lang po, {stock} na lang ang natitira sa {product}. Ayaw ko pong mag-imbento — "
        "kung gusto ninyo, i-reserve na natin.",
    ],
}


# ==========================================================
# 🛡️ GUARDRAILS
# ==========================================================
FAKE_PATTERNS = {
    "fake_scarcity": [
        r"(\d+)\s*(na lang|lang)\s*(po)?\s*(ang)?\s*(stock|natitira)",
        r"sold out na.*(pero|may)\s*extra",
    ],
    "fake_review": [
        r"(ayon sa|sabi ng)\s*\d+\s*(buyers?|customers?)",
        r"viral\s*(na|ang)\s*(review|testimonial)",
    ],
    "fake_health": [
        r"(gagamot|cure|heal|siguradong gagaling)",
        r"(FDA|DOH)\s*(approved)",
    ],
    "fake_price": [
        r"(original price|dating presyo)\s*(₱?\d+)\s*(ngayon|pero)",
    ],
    "manipulation": [
        r"(bili na|order na).*(pagsisisi|maiiyak)",
        r"(last chance|huling tawag)\s*(na)?",
    ],
}


# ==========================================================
# 💾 PERSISTENCE
# ==========================================================
def load_data():
    defaults = {
        "store_name": None, "owner_name": None, "live_platform": None,
        "capital": 0, "monthly_expenses": 0,
        "shipping_fee": 0, "min_free_ship": 0, "return_policy": None,
        "products": [], "sales_log": [], "live_log": [],
        "take_over_active": False, "last_activity": None,
    }
    if DATA_FILE.exists():
        with open(DATA_FILE) as f:
            data = json.load(f)
        for k, v in defaults.items():
            data.setdefault(k, v)
        return data
    return defaults


def save_data(data):
    data["last_activity"] = datetime.now().isoformat()
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def reset_tindahan():
    if DATA_FILE.exists():
        DATA_FILE.unlink()
    print("✅ Na-reset. Fresh start!")


# ==========================================================
# 🚀 ONBOARDING
# ==========================================================
def next_onboarding_question():
    data = load_data()
    for q in ONBOARDING_QUESTIONS:
        if data.get(q["key"]) in (None, "", 0):
            return q
    return None


def apply_onboarding_answer(key, value):
    data = load_data()
    if key in ("capital", "monthly_expenses", "shipping_fee", "min_free_ship"):
        try:
            value = float(value)
            MathLogic._validate_amount(value, key)
        except (ValueError, TypeError) as e:
            return {"ok": False, "error": f"Numero lang po: {e}"}
    data[key] = value
    save_data(data)
    return {"ok": True, "data": data}


# ==========================================================
# 🎮 TAKE-OVER
# ==========================================================
def detect_take_over(text: str) -> str:
    t = text.lower()
    if any(re.search(p, t) for p in HAND_BACK_TRIGGERS):
        return "hand_back"
    if any(re.search(p, t) for p in TAKE_OVER_TRIGGERS):
        return "take_over"
    return "none"


def set_take_over(active: bool):
    data = load_data()
    data["take_over_active"] = active
    save_data(data)


# ==========================================================
# 🎬 SCRIPT RENDERER
# ==========================================================
def render_script(key, **kwargs):
    data = load_data()
    ctx = {
        "store": data["store_name"] or "ating shop",
        "owner": data["owner_name"] or "seller",
        "agent": "AI Co-host",
        "shipping": data.get("shipping_fee", 0),
        "min_amount": data.get("min_free_ship", 0),
        "ship_fee": data.get("shipping_fee", 0),
        "buyer": "boss",
        **kwargs,
    }
    templates = LIVE_SCRIPTS.get(key, [])
    if not templates:
        return None
    try:
        return random.choice(templates).format(**ctx)
    except KeyError as e:
        return f"[script '{key}' kulang ng field: {e}]"


# ==========================================================
# 🛡️ GUARDRAILS
# ==========================================================
def check_guardrails(text: str) -> dict:
    issues = []
    t = text.lower()
    for category, patterns in FAKE_PATTERNS.items():
        for p in patterns:
            if re.search(p, t):
                issues.append({"type": category, "matched": p})
    return {"ok": len(issues) == 0, "issues": issues}


# ==========================================================
# 📦 PRODUCTS
# ==========================================================
PRODUCT_SYNONYMS = {
    "coke": "Coke 1.5L", "coca cola": "Coke 1.5L", "softdrinks": "Coke 1.5L",
    "royal": "Royal 1.5L", "c2": "C2 Green Tea",
    "tubig": "Bottled Water 500ml", "water": "Bottled Water 500ml",
    "kape": "Kape 3-in-1", "pancit": "Lucky Me Pancit Canton",
    "sardinas": "555 Sardinas", "corned beef": "Argentina Corned Beef",
    "chips": "Piattos", "skyflakes": "Skyflakes", "sabon": "Safeguard Sabon",
    "shampoo": "Palmolive Shampoo", "bigas": "Bigas (per kilo)",
    "asukal": "Asukal (per kilo)", "itlog": "Itlog",
    "load": "Globe Load 20", "globe": "Globe Load 20", "smart": "Smart Load 20",
    "gcash": "GCash Cash-in",
}


def find_product(text: str):
    data = load_data()
    t = text.lower()
    for p in data["products"]:
        if p["name"].lower() in t:
            return p
    for syn, prod_name in PRODUCT_SYNONYMS.items():
        if re.search(r'\b' + re.escape(syn) + r'\b', t):
            for p in data["products"]:
                if p["name"].lower() == prod_name.lower():
                    return p
    for p in data["products"]:
        first = p["name"].lower().split()[0]
        if first and re.search(r'\b' + re.escape(first) + r'\b', t):
            return p
    return None


ADD_RE = re.compile(
    r'ADD:\s*(?P<name>[^|]+?)\s*\|\s*(?P<buy>\d+(?:\.\d+)?)\s*\|\s*'
    r'(?P<sell>\d+(?:\.\d+)?)\s*\|\s*(?P<stock>\d+)\s*\|\s*(?P<unit>[a-zA-Z]+)',
    re.I,
)


def parse_add_commands(text):
    results = []
    for m in ADD_RE.finditer(text):
        try:
            buy = MathLogic._validate_amount(float(m.group("buy")), "buy")
            sell = MathLogic._validate_amount(float(m.group("sell")), "sell")
            results.append({
                "name": m.group("name").strip(),
                "buy_price": buy,
                "sell_price": sell,
                "stock": int(m.group("stock")),
                "unit": m.group("unit").lower(),
            })
        except ValueError:
            continue
    return results


def add_products(products):
    data = load_data()
    for p in products:
        existing = next(
            (x for x in data["products"] if x["name"].lower() == p["name"].lower()),
            None,
        )
        if existing:
            existing.update(p)
        else:
            data["products"].append(p)
    save_data(data)
    return {"ok": True, "count": len(products), "products": data["products"]}


# ==========================================================
# 🛒 SALES
# ==========================================================
SELL_RE = re.compile(r'SELL:\s*(?P<name>[^|]+?)\s*\|\s*(?P<qty>\d+)', re.I)


def parse_sell_commands(text):
    return [{"name": m.group("name").strip(), "qty": int(m.group("qty"))}
            for m in SELL_RE.finditer(text)]


def record_sales(sales):
    data = load_data()
    results = []
    for s in sales:
        prod = next(
            (p for p in data["products"] if p["name"].lower() == s["name"].lower()),
            None,
        )
        if not prod:
            results.append({"ok": False, "reason": f"Wala sa inventory: {s['name']}"})
            continue
        if prod["stock"] < s["qty"]:
            results.append({"ok": False,
                            "reason": f"Kulang stock ng {prod['name']} ({prod['stock']} lang)"})
            continue
        try:
            gross, _ = MathLogic.compute_total(prod["sell_price"], s["qty"])
            cost, _ = MathLogic.compute_total(prod["buy_price"], s["qty"])
        except ValueError as e:
            results.append({"ok": False, "reason": str(e)})
            continue
        profit = MathLogic.round_money(gross - cost)
        prod["stock"] -= s["qty"]
        data["sales_log"].append({
            "product": prod["name"], "qty": s["qty"],
            "gross": gross, "cost": cost, "profit": profit,
            "timestamp": datetime.now().isoformat(),
        })
        results.append({
            "ok": True, "product": prod["name"], "qty": s["qty"],
            "gross": gross, "cost": cost, "profit": profit,
        })
    save_data(data)
    return {"ok": True, "results": results}


# ==========================================================
# 📊 FINANCIAL REPORT
# ==========================================================
def financial_report():
    data = load_data()
    total_gross = MathLogic.round_money(sum(s["gross"] for s in data["sales_log"]))
    total_cost = MathLogic.round_money(sum(s["cost"] for s in data["sales_log"]))
    gross_profit = MathLogic.round_money(total_gross - total_cost)
    net_profit = MathLogic.round_money(gross_profit - data["monthly_expenses"])
    capital = data["capital"]
    roi = MathLogic.compute_roi(net_profit, capital)
    avg_margin_pct = (gross_profit / total_gross * 100) if total_gross > 0 else 0
    bep = MathLogic.compute_break_even(data["monthly_expenses"], avg_margin_pct)

    return {
        "store_name": data["store_name"],
        "owner_name": data["owner_name"],
        "total_gross": total_gross,
        "total_cost": total_cost,
        "gross_profit": gross_profit,
        "monthly_expenses": data["monthly_expenses"],
        "net_profit": net_profit,
        "capital": capital,
        "roi_percent": roi,
        "avg_margin_pct": round(avg_margin_pct, 2),
        "break_even_sales_php": bep if bep is not None else 0,
        "inventory_value": MathLogic.round_money(
            sum(p["buy_price"] * p["stock"] for p in data["products"])
        ),
        "low_stock": [p["name"] for p in data["products"] if p["stock"] <= 5],
        "number_of_products": len(data["products"]),
    }


def summary_for_ai():
    data = load_data()
    r = financial_report()
    return "\n".join([
        f"SHOP: {r['store_name']} | SELLER: {r['owner_name']} | PLATFORM: {data.get('live_platform', 'N/A')}",
        f"TAKE-OVER ACTIVE: {data.get('take_over_active', False)}",
        f"Puhunan: ₱{r['capital']:.2f} | Buwanang gastos: ₱{r['monthly_expenses']:.2f}",
        f"Shipping: ₱{data.get('shipping_fee', 0):.2f} | Free ship min: ₱{data.get('min_free_ship', 0):.2f}",
        f"Return policy: {data.get('return_policy', 'N/A')}",
        f"Produkto: {r['number_of_products']} | Inventory value: ₱{r['inventory_value']:.2f}",
        f"Gross sales: ₱{r['total_gross']:.2f} | COGS: ₱{r['total_cost']:.2f}",
        f"Gross profit: ₱{r['gross_profit']:.2f} | Net profit: ₱{r['net_profit']:.2f}",
        f"ROI: {r['roi_percent']}% | Avg margin: {r['avg_margin_pct']}%",
        f"BEP (buwan): ₱{r['break_even_sales_php']:.2f}",
        f"Low stock: {', '.join(r['low_stock']) or 'wala'}",
    ])


def cashflow_advice():
    r = financial_report()
    advice = []
    if r["total_gross"] == 0:
        advice.append("Wala pa tayong benta. Simulan sa 'SELL: pangalan | qty' kada order.")
    if r["gross_profit"] > 0 and r["net_profit"] < 0:
        advice.append(
            f"Kumikita sa produkto (₱{r['gross_profit']:.2f}) pero lugi sa gastos "
            f"(₱{r['net_profit']:.2f}). Taasan ang benta o bawasan ang gastos."
        )
    if r["net_profit"] > 0:
        advice.append(f"Net profit ₱{r['net_profit']:.2f}, ROI {r['roi_percent']}%. Ipagpatuloy!")
    if 0 < r["roi_percent"] < 5:
        advice.append("Mababa ang ROI (<5%). Tingnan ang margin per product.")
    if r["low_stock"]:
        advice.append(f"Malapit nang maubos: {', '.join(r['low_stock'])}. I-restock na.")
    if r["break_even_sales_php"] > 0:
        advice.append(f"Break-even ngayong buwan: ₱{r['break_even_sales_php']:.2f} na benta.")
    return "\n".join(advice) if advice else "Tuloy lang, i-track natin bawat benta."


print("✅ Cell 2 done. Core system ready.")

✅ Cell 2 done. Core system ready.


In [4]:
# ============================================================
# CELL 3 — SeaLLM + API + seguridad (kailangan ang Cell 2)
# ============================================================
import os
import re
import threading
import secrets as pysecrets
import io
import edge_tts
from typing import Any, Dict, List, Optional
from fastapi import FastAPI, Header, HTTPException, Response
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from llama_cpp import Llama

MODEL_PATH = "./seallm/SeaLLM-7B-v2.q4_0.gguf"
assert os.path.exists(MODEL_PATH), "Wala ang model. Patakbuhin muna ang download cell."

print("⏳ Nilo-load ang SeaLLM... (1-3 minuto)")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_threads=os.cpu_count() or 4,
    verbose=False,
)
print("✅ SeaLLM loaded.")

STATE_LOCK = threading.RLock()
MODEL_NAME = "SeaLLM-7B-v2-q4_0"
AGENT_NAME = "INDAY"


# ==========================================================
# SELLER TOKEN (galing sa .env file — standalone, walang Kaggle Secrets)
# ==========================================================
ENV_FILE = os.path.join(os.getcwd(), ".env")


def _read_env_file(path=ENV_FILE):
    """Basahin ang .env nang walang dagdag na dependency."""
    values = {}
    if not os.path.exists(path):
        return values
    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            values[key.strip()] = val.strip().strip('"').strip("'")
    return values


def _load_seller_token():
    """Priority: os.environ, tapos ang .env file sa tabi ng notebook."""
    t = os.environ.get("SELLER_TOKEN", "").strip()
    if t:
        return t
    return _read_env_file().get("SELLER_TOKEN", "").strip()


def _ensure_env_template():
    """Gumawa ng .env.example para madaling i-setup, at i-ignore ang .env sa git."""
    example = os.path.join(os.getcwd(), ".env.example")
    if not os.path.exists(example):
        with open(example, "w", encoding="utf-8") as fh:
            fh.write(
                "# Kopyahin ito sa .env at lagyan ng totoong halaga.\n"
                "# Huwag i-commit ang .env.\n"
                "SELLER_TOKEN=palitan_ito_ng_16_characters_pataas\n"
                "# edge-tts is free and needs no key. Change the voice here.\n"
                "TTS_VOICE=fil-PH-BlessicaNeural\n"
            )
    ignore = os.path.join(os.getcwd(), ".gitignore")
    existing = ""
    if os.path.exists(ignore):
        existing = open(ignore, encoding="utf-8").read()
    # Ignore only the secret and the model weights, never the notebook itself.
    wanted = [".env", "*.gguf", "cloudflared", "tunnel.log", "tindahan_data.json"]
    missing = [w for w in wanted if w not in existing.split()]
    if missing:
        with open(ignore, "a", encoding="utf-8") as fh:
            prefix = "" if not existing or existing.endswith("\n") else "\n"
            fh.write(prefix + "\n".join(missing) + "\n")


# edge-tts: free Microsoft Edge voices, no API key, no billing.
# fil-PH-BlessicaNeural is the Filipino female voice closest to INDAY.
TTS_VOICE = os.environ.get("TTS_VOICE", "").strip() or _read_env_file().get(
    "TTS_VOICE", ""
).strip() or "fil-PH-BlessicaNeural"

SELLER_TOKEN = _load_seller_token()
if len(SELLER_TOKEN) < 16:
    SELLER_TOKEN = ""
    print("⚠️ Walang SELLER_TOKEN (min. 16 characters). Naka-disable ang seller commands.")
else:
    print("✅ Seller token loaded.")


def is_seller(header_value):
    if not SELLER_TOKEN or not header_value:
        return False
    return pysecrets.compare_digest(header_value.encode(), SELLER_TOKEN.encode())


# ==========================================================
# AI PROMPT
# ==========================================================
SALES_AGENT_SYSTEM = (
    f"Ikaw si {AGENT_NAME}, isang tindera at AI co-host ng online shop sa Pilipinas.\n\n"
    "WIKA: Sumagot sa natural na Filipino/Tagalog, parang totoong tindera. "
    "Gamitin ang 'boss', 'sir', 'ma'am', 'sulit', 'legit' kapag bagay.\n\n"
    "TAPAT: Gamitin lang ang presyo, stock, shipping, at policy mula sa DATOS. Huwag mag-imbento. "
    "Kung hindi alam, sabihin: 'Titingnan ko po, boss.'\n\n"
    "BAWAL: fake scarcity, fake reviews, health claims na walang FDA, at panlilinlang.\n\n"
    "DISCLOSURE: Kung tinanong, sabihin: 'Ako ay AI assistant. Ang seller ang may final approval.'"
)


# ==========================================================
# STORY BLOCKS 1-15
# Source: Convai character 9a6d5a76-19b0-4b66-bc2e-d76e13e85795,
# embedded below, exported from seallm/inday_story_blocks.json. Same graph
# the React app uses, so both sides route identically.
# ==========================================================
STORY_BLOCKS = json.loads(r"""[
 {
  "n": 1,
  "name": "Masiglang Pagbati",
  "sectionId": "35954fb7-ebf2-4926-8cd3-f6c522576cb7",
  "objective": "Ikaw si Inday, isang masayahin at chismosang tindera ng sari-sari store. Batiin nang masigla ang kapitbahay sa natural na Taglish at magbiro tungkol sa sobrang init ng panahon ngayon. Tanungin siya kung naghahanap ba siya ng pampalamig o meryenda. Hanggang 2 maikling pangungusap lang at bawal gumamit ng brackets o emojis.",
  "decisions": [
   {
    "when": "Sumagot ang kausap o nagtanong kung ano ang paninda",
    "goto": 2
   },
   {
    "when": "Nagtanong ang kausap kung pwedeng umutang, magpalista, magpa-load, o mag-GCash",
    "goto": 5
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 2,
  "name": "Alok na Paninda",
  "sectionId": "638266d9-4267-47a9-aef8-23f71cc6f3be",
  "objective": "Kung may partikular na paninda o pagkain na binanggit ang kausap, iyon agad ang ipagmalaki at ibida sa masiglang Taglish. Kung wala pa siyang binanggit na partikular na item, ialok ang mga bagong dating at mabentang paninda sa sari-sari store tulad ng malamig na softdrinks, kape, snacks, lutong meryenda, o pang-araw-araw na kailangan sa bahay. Sabihin kung gaano kasulit at kasariwa ang mga paninda rito. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nagtanong ang kausap kung magkano o kung may stock pa",
    "goto": 3
   },
   {
    "when": "Sinabi ng kausap na bibili na siya agad",
    "goto": 4
   },
   {
    "when": "Nagtanong ang kausap kung pwedeng umutang, magpalista, magpa-load, o mag-GCash",
    "goto": 5
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   },
   {
    "when": "Kulang ang ibinigay na bayad ng bumibili o umalis nang hindi nakakapagbayad.",
    "goto": 15
   }
  ]
 },
 {
  "n": 3,
  "name": "Presyo at Pang-akit",
  "sectionId": "30fb2322-434a-40f9-9f06-e022d11c0943",
  "objective": "Sabihin sa masiglang Taglish na kumpleto ang stock at sobrang abot-kaya ng presyo kaya swak na swak sa budget ng pamilya. Banggitin na mabilis maubos ang mga paninda dahil dito bumibili ang buong barangay araw-araw. Hanggang 2 pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nagtanong ang kausap kung pwedeng umutang, magpalista, magpa-load, o mag-GCash",
    "goto": 5
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 4,
  "name": "Imbitasyon at Paalam",
  "sectionId": "323d5475-40e6-43e8-a718-cd86e8c2b3c0",
  "objective": "Yayain na ang kapitbahay na dumaan at bumili na ngayon sa tindahan bago pa maubusan ng stock. Magpasalamat nang masigla sa Taglish at sabihing ipaghahanda mo na ang bibilhin niya. Hanggang 2 pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 5,
  "name": "Utang at GCash",
  "sectionId": "d35b7d94-d187-4055-b47d-d6789a6f0387",
  "objective": "Kung nagtatanong tungkol sa GCash o E-Load, sabihin sa masiglang Taglish na pwedeng-pwede mag-cash in, cash out, at magpa-load dito nang mabilis at maliit lang ang patong. Kung nagtatanong naman kung pwedeng umutang o magpalista, idaan sa magalang na biro na cash o GCash muna ngayon para tuloy-tuloy ang ikot ng puhunan ng tindahan, o kausapin muna si Ate o Kuya na may-ari. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Pumayag ang kausap o sinabing pupunta na sa tindahan",
    "goto": 4
   },
   {
    "when": "Nagtanong ulit ang kausap tungkol sa ibang paninda o presyo",
    "goto": 3
   },
   {
    "when": "Lumihis sa ibang topic o nag-iba ng kwento ang kausap",
    "goto": 8
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 6,
  "name": "Panukli at Pasuyo",
  "sectionId": "e1f15cce-65f7-446a-b2fe-029cdae53407",
  "objective": "Kung nagtatanong kung may panukli sa malaking pera tulad ng 500 o 1000 pesos, magalang na paalalahanan sa masiglang Taglish na magdala ng barya o smaller bills kung kakayanin, o magbayad na lang via GCash para walang problema sa sukli. Kung nagtatanong kung may delivery o pasuyo, sabihin na dumaan muna sa tindahan o i-message si Ate o Kuya kung maramihan ang bibilhin. Hanggang 2 pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nagtanong ang kausap kung may panukli sa buong pera o kung pwedeng ipa-deliver",
    "goto": 4
   },
   {
    "when": "May barya na o magbabayad na ngayon",
    "goto": 7
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   }
  ]
 },
 {
  "n": 7,
  "name": "Bayaran at Sukli",
  "sectionId": "48f10dd1-625f-4644-90d5-a35dcb6be7ed",
  "objective": "Asikasuhin sa masiglang Taglish ang aktwal na palitan ng pera—tanggapin ang bayad sa cash o GCash, sabihing eksakto ang sukli kung may sukli man, at iabot ang biniling paninda. Tanungin kung may pahabol pa siyang gustong idagdag bago umalis. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Wala nang idaragdag ang kausap at tapos na ang bayaran",
    "goto": 4
   },
   {
    "when": "May pahabol pang gustong bilhin ang kausap",
    "goto": 2
   },
   {
    "when": "Nagbintang o nanira na nagpadala ng pera sa GCash o e-wallet at tinanggap daw ito, o nag-akusa nang walang patunay",
    "goto": 10
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 8,
  "name": "Off-Topic at Balik-Benta",
  "sectionId": "63199ca3-57fd-42f6-9f21-037992f155de",
  "objective": "Kung lumihis sa ibang topic o kwento ang kausap na wala sa paninda ng tindahan, makisakay sandali nang magalang at masayahin sa natural na Taglish na parang kapitbahay, pero agad na ibalik ang usapan sa mga mabentang paninda o pangangailangan ng kausap sa tindahan. Huwag magbigay ng payo sa sensitibong isyu, pulitika, o malisyosong tsismis. Hanggang 2 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Sumakay ang kausap at nagtanong tungkol sa paninda o presyo",
    "goto": 2
   },
   {
    "when": "Nagpaalam na ang kausap o nagpasalamat na",
    "goto": 4
   }
  ]
 },
 {
  "n": 9,
  "name": "Iwas-Scam at OTP",
  "sectionId": "a167e50f-2a6e-48f4-9765-54943f20e133",
  "objective": "Kung may humihingi ng OTP, MPIN, PIN, password, o sensitibong detalye ng e-wallet tulad ng GCash o Maya at bank account, o nagpapanggap na nagmamadali upang manloko o manglamang sa tindahan, mariin pero magalang na tanggihan ito sa natural na Taglish. Malinaw na sabihin na bawal ibigay ang OTP, PIN, at account details kahit kanino, at ipaalala na mag-ingat sa scammer at kumpirmahin muna sa may-ari ng tindahan kung may kahina-hinalang transaksyon. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Umatras ang kausap o nagpaalam na",
    "goto": 4
   },
   {
    "when": "Bumalik sa totoong pagbili o lehitimong transaksyon sa tindahan",
    "goto": 2
   }
  ]
 },
 {
  "n": 10,
  "name": "Depensa sa Maling Paratang",
  "sectionId": "de177a55-e31d-471b-9e57-12075c69d9b6",
  "objective": "Kung pinaparatangan ka o ang tindahan na nakatanggap ng bayad sa GCash o e-wallet nang wala namang pumasok na pera, o sinisiraan ka at ang may-ari, manatiling kalmado, magalang, pero matatag sa natural na Taglish. Malinaw na sabihin na walang naitalang bayad sa account ng tindahan at hindi ka awtorisadong magkumpirma ng pumasok na pera nang walang beripikasyon. Ipaalala na i-check ang opisyal na e-wallet history at kausapin nang maayos ang may-ari ng tindahan, at huwag gumanti ng bastos o mapanirang salita. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nagpakita ng resibo o handang makipag-ugnayan nang maayos sa may-ari",
    "goto": 7
   },
   {
    "when": "Patuloy na naninira o nang-aakusa nang walang patunay",
    "goto": 4
   },
   {
    "when": "May lasing, nambabastos, nangbu-bully, banta ng sunog, o matinding gulo at pangangailangan ng tulong pangkaligtasan",
    "goto": 11
   }
  ]
 },
 {
  "n": 11,
  "name": "Kaligtasan at Emergency",
  "sectionId": "fd2f3681-f0bf-4402-9587-c6b90c8460a1",
  "objective": "Kung may lasing, nambabastos sa tindera, nangbu-bully sa tindahan, banta ng sunog, o matinding gulo, manatiling kalmado, matatag, at malinaw sa natural na Taglish nang hindi kinukunsinti ang pang-aabuso o pananakit. Unahin ang kaligtasan ng tindera at may-ari—ipaalala na lumayo kung delikado, isara o i-lock ang tindahan kung kailangan, at makipag-ugnayan sa may-ari, opisyal na barangay hotline, pulis o 911, bumbero kung may sunog, o lehitimong law office. Huwag mag-imbento ng telepono o numero kung wala sa opisyal na talaan ng tindahan, at huwag magbigay ng pormal na legal na hatol. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Humupa ang gulo o umatras ang nanggugulo",
    "goto": 2
   },
   {
    "when": "Bumalik sa normal na usapan sa tindahan",
    "goto": 2
   }
  ]
 },
 {
  "n": 12,
  "name": "Kontak ng Barangay at Emergency",
  "sectionId": "9f41519b-1206-4e48-b242-66143746118b",
  "objective": "Kung humihingi ng contact number ng barangay sa Bacoor, pulis, bumbero, o emergency hotline, ibigay nang malinaw at kalmado sa natural na Taglish ang naaangkop na lehitimong numero mula sa opisyal na talaan o Knowledge Bank (tulad ng Bacoor 161, BDRRMO 046-417-0727, PNP Bacoor, o BFP Bacoor). Kung walang nakalistang direktang numero ang partikular na barangay, sabihin nang tapat na gamitin muna ang Bacoor 161, 911, o Barangay Affairs (046-436-3540) at huwag mag-imbento ng numero. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": []
 },
 {
  "n": 13,
  "name": "Kontak ng Supplier",
  "sectionId": "e1782069-b0f7-4c77-97aa-77db49d54596",
  "objective": "Kung nagtatanong o nagbabahagi ng contact ng supplier ng paninda (halimbawa ay Rex para sa Yelo o Ice Tube, numero 0920240314), ibigay nang malinaw at masigla sa natural na Taglish ang pangalan ng supplier, paninda, at contact number. Tanungin ang may-ari ng sari-sari store kung gusto niyang i-save o i-update ang supplier contact na ito sa talaan ng tindahan, at ipaalala na siguraduhing tama ang numero bago i-save. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": []
 },
 {
  "n": 14,
  "name": "Sukli at Sobra sa Bayad",
  "sectionId": "539f5b63-f1cb-4b3e-b891-96580888771a",
  "objective": "Maging tapat, kalmado, at mabilis sa pag-aayos ng bayad sa natural na Taglish. Kung kulang ang sukli, humingi agad ng paumanhin at sabihing bibilangin nang maayos para maibigay ang kulang; kung sobra ang bayad o nakalimutan ang sukli, agad itong ipaalam at isoli nang buo sa bumibili. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Nakuha na ng bumibili ang tamang sukli, naisoli na ang sobra, o tapos na ang usapan.",
    "goto": 2
   },
   {
    "when": "Nagagalit pa rin o nag-aakusa ang bumibili na sinadyang dayain ang sukli niya.",
    "goto": 10
   },
   {
    "when": "Kulang ang ibinigay na bayad ng bumibili o umalis nang hindi nakakapagbayad.",
    "goto": 15
   }
  ]
 },
 {
  "n": 15,
  "name": "Kulang o Hindi Nagbayad",
  "sectionId": "fb2fb776-579e-4167-b8f7-93c6f6848183",
  "objective": "Kalmado at magalang na ipaalam sa bumibili sa natural na Taglish na kulang pa ang bayad niya o nakalimutang magbayad bago umalis. Ipaalala ang eksaktong halaga nang walang pagbibintang, at tanungin kung dadagdagan niya ang bayad, babawasan ang binili, o ililista na lang kung pinapayagan. Hanggang 2 hanggang 3 maikling pangungusap lang at bawal ang brackets.",
  "decisions": [
   {
    "when": "Dinagdagan ng bumibili ang kulang o nagbayad na nang buo.",
    "goto": 4
   },
   {
    "when": "Nagpasyang magbawas o magpalit ng paninda para magkasya ang pera niya.",
    "goto": 2
   },
   {
    "when": "Nakiusap na ilista o utangin muna ang kulang.",
    "goto": 5
   },
   {
    "when": "Nagalit, tumangging magbayad, o nag-akusa na nagbayad na raw siya.",
    "goto": 10
   }
  ]
 }
]""")
STORY_BY_N = {b["n"]: b for b in STORY_BLOCKS}
CONVAI_CHARACTER_ID = "9a6d5a76-19b0-4b66-bc2e-d76e13e85795"

# Salita na masyadong karaniwan para maging trigger.
_STOP = {
    "sumagot", "nagtanong", "nagbintang", "kung", "ang", "ng", "sa", "na",
    "at", "o", "siya", "ito", "iyon", "para", "daw", "ba", "po", "may",
}


def route_story(text, current_n):
    """Follow the Convai decision graph: first rule whose words appear in text."""
    block = STORY_BY_N.get(current_n) or STORY_BY_N[1]
    lower = (text or "").lower()
    for d in block["decisions"]:
        target = d.get("goto")
        if not target or target == current_n:
            continue
        words = [w for w in re.findall(r"[a-zA-ZñÑ]{4,}", d["when"].lower())
                 if w not in _STOP]
        if any(w in lower for w in words):
            return target
    return None


def block_objective(n):
    b = STORY_BY_N.get(n) or STORY_BY_N[1]
    return b["objective"]


def _tidy(text, max_chars=320):
    """Keep at most two full sentences, so speech never stops mid-word."""
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[.!?])\s+", text)
    out = ""
    for p in parts:
        if out and len(out) + len(p) + 1 > max_chars:
            break
        out = (out + " " + p).strip()
    if out and not out.endswith((".", "!", "?")):
        out = out.rsplit(" ", 1)[0].rstrip(",;:") + "."
    return out or text


def _dedupe(text, max_repeats=1):
    """Drop sentences the model emitted more than once."""
    parts = re.split(r"(?<=[.!?])\s+", text)
    seen, out = {}, []
    for p in parts:
        key = p.strip().lower()
        if not key:
            continue
        seen[key] = seen.get(key, 0) + 1
        if seen[key] <= max_repeats:
            out.append(p.strip())
    return " ".join(out)


def llm_reply(user_text, context, max_tokens=300, temperature=0.6, story_n=1):
    # The active Convai block's objective is the instruction Convai itself
    # would send, so INDAY stays in character per scene.
    system = (SALES_AGENT_SYSTEM
              + f"\n\nSTORY BLOCK {story_n}: {block_objective(story_n)}"
              + "\n\nDATOS NG TINDAHAN:\n" + context)
    out = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        max_tokens=min(max_tokens, 160),
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.18,
        frequency_penalty=0.35,
        presence_penalty=0.2,
        stop=["<|im_end|>", "</s>", "\n\nBoss,"],
    )
    return _tidy(_dedupe(out["choices"][0]["message"]["content"].strip()))


# ==========================================================
# BUYER CONTEXT: walang kapital, gastos, o kita
# ==========================================================
def buyer_context():
    d = load_data()
    lines = [
        f"Tindahan: {d.get('store_name') or 'ating shop'}",
        f"Shipping: ₱{d.get('shipping_fee', 0):.2f}",
        f"Free shipping mula sa: ₱{d.get('min_free_ship', 0):.2f}",
        f"Return policy: {d.get('return_policy') or 'wala pang nakatakda'}",
        "Produkto:",
    ]
    for p in d["products"]:
        lines.append(f"- {p['name']}: ₱{p['sell_price']:.2f}, stock {p['stock']} {p['unit']}")
    if not d["products"]:
        lines.append("- wala pang produkto")
    return "\n".join(lines)


def all_numbers(text):
    return {round(float(x.replace(",", "")), 2)
            for x in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}


def numbers_grounded(answer):
    return all_numbers(answer) <= all_numbers(buyer_context())


def _ground_numbers(answer):
    """Drop only the sentences that quote a figure the store data cannot back."""
    allowed = all_numbers(buyer_context())
    kept = []
    for sentence in re.split(r"(?<=[.!?])\s+", answer):
        if not sentence.strip():
            continue
        if all_numbers(sentence) <= allowed:
            kept.append(sentence.strip())
    if not kept:
        return "Titingnan ko po muna, boss, para sigurado. Ang seller ang may final approval."
    return " ".join(kept)


# ==========================================================
# PATTERNS
# ==========================================================
PRICE_RE = re.compile(r"\b(magkano|presyo|mahal|price)\b", re.I)
STOCK_RE = re.compile(r"\b(stock|natitira|available|meron pa|may stock)\b", re.I)
POLICY_RE = re.compile(r"\b(shipping|ship|delivery|cod|gcash|maya|bayad|return|palit|policy)\b", re.I)
SENSITIVE_RE = re.compile(
    r"\b(refund|reklamo|sira|kaso|demanda|legal|sakit|gamot|scam|niloko|galit)\b", re.I
)

# Case-insensitive na ang take-over (ayos sa bug ng Cell 2)
TAKE_OVER_PATTERNS = [re.compile(p, re.I) for p in [
    r"\bai[, ]+ikaw na( muna)?\b",
    r"\btake over\b",
    r"\bai[, ]+sagutin mo\b",
    r"\bikaw na muna\b",
]]
HAND_BACK_PATTERNS = [re.compile(p, re.I) for p in [
    r"\bako na\b",
    r"\bseller na\b",
    r"\bhand back\b",
    r"\bstop take over\b",
]]


def take_over_cmd(text):
    if any(p.search(text) for p in HAND_BACK_PATTERNS):
        return "hand_back"
    if any(p.search(text) for p in TAKE_OVER_PATTERNS):
        return "take_over"
    return None


# ==========================================================
# BUYER: read-only
# ==========================================================
def buyer_reply(text, max_tokens=300, temperature=0.6, story_n=1):
    data = load_data()
    if not data.get("setup_done"):
        return "Hindi pa po bukas ang tindahan, boss. Seller pa ang nag-se-setup."
    if not data.get("take_over_active"):
        return "Seller po ang sumasagot ngayon, boss. Sandali lang po."
    if not text:
        return "Ano po ang hanap ninyo, boss?"
    if SENSITIVE_RE.search(text):
        return "Ipaubaya ko po ito sa seller, boss. Siya na po ang sasagot."

    if not POLICY_RE.search(text) and (PRICE_RE.search(text) or STOCK_RE.search(text)):
        p = find_product(text)
        if not p:
            return "Wala po sa listahan namin ang produktong 'yan, boss. Titingnan ko po sa seller."
        if PRICE_RE.search(text):
            return f"₱{p['sell_price']:.2f} po ang {p['name']}, boss."
        if p["stock"] <= 0:
            return f"Sold out na po ang {p['name']}, boss."
        return f"Meron pa pong {p['stock']} {p['unit']} ng {p['name']}, boss."

    answer = llm_reply(text, buyer_context(), max_tokens, temperature, story_n)
    if not check_guardrails(answer)["ok"]:
        return "Titingnan ko po muna, boss, para sigurado. Ang seller ang may final approval."
    return _ground_numbers(answer)


# ==========================================================
# SELLER: may token
# ==========================================================
def take_over_reply(text):
    cmd = take_over_cmd(text)
    if cmd == "take_over":
        set_take_over(True)
        return "AI na po muna. Presyo at stock lang mula sa listahan. Sabihin ang 'ako na' para maibalik."
    if cmd == "hand_back":
        set_take_over(False)
        return "Okay, seller na ang bahala. Salamat po."
    return None


def add_reply(text):
    parsed = parse_add_commands(text)
    if not parsed:
        return "Hindi ko naintindihan. " + PRODUCT_INPUT_PROMPT
    add_products(parsed)
    lines = [
        f"• {p['name']}: bili ₱{p['buy_price']:.2f}, benta ₱{p['sell_price']:.2f}, "
        f"stock {p['stock']} {p['unit']}"
        for p in parsed
    ]
    return "Naidagdag na po:\n" + "\n".join(lines)


def sell_reply(text):
    sales = parse_sell_commands(text)
    if not sales:
        return "Hindi ko naintindihan ang benta. Gamitin: SELL: pangalan | dami"
    results = record_sales(sales)["results"]
    lines = []
    for r in results:
        if r["ok"]:
            lines.append(f"✅ {r['product']} x{r['qty']}: ₱{r['gross']:.2f} (kita ₱{r['profit']:.2f})")
        else:
            lines.append(f"❌ {r['reason']}")
    return "\n".join(lines)


# ==========================================================
# ONBOARDING (seller lang)
# ==========================================================
def onboarding_next(text):
    data = load_data()
    if data.get("setup_done"):
        return None

    text = (text or "").strip()
    pending = data.get("pending_key")

    # Unang mensahe: i-type ang "mulai" para simulan
    if pending is None:
        if text.lower() in ("mulai", "start", "setup"):
            q = ONBOARDING_QUESTIONS[0]
            data["pending_key"] = q["key"]
            save_data(data)
            return q["question"]
        return "I-type po ang 'mulai' para simulan ang setup."

    q = next(x for x in ONBOARDING_QUESTIONS if x["key"] == pending)
    if not text:
        return q["question"]

    if q["type"] == "number":
        clean = text.replace(",", "")
        if not re.fullmatch(r"\d+(?:\.\d+)?", clean):
            return "Numero lang po. " + q["question"]
        value = float(clean)
    else:
        value = text

    # Ang sagot ay sa tanong na kasalukuyang nakabinbin
    data[q["key"]] = value

    keys = [x["key"] for x in ONBOARDING_QUESTIONS]
    idx = keys.index(q["key"])
    if idx + 1 < len(ONBOARDING_QUESTIONS):
        nxt = ONBOARDING_QUESTIONS[idx + 1]
        data["pending_key"] = nxt["key"]
        save_data(data)
        return nxt["question"]

    data["pending_key"] = None
    data["setup_done"] = True
    save_data(data)
    return f"Tapos na ang setup!\n\n{summary_for_ai()}\n\n{PRODUCT_INPUT_PROMPT}"


def seller_reply(text):
    up = text.upper()

    reply = take_over_reply(text)
    if reply:
        return reply

    if not load_data().get("setup_done"):
        if "ADD:" in up or "SELL:" in up:
            return "Tapusin muna po ang setup ng tindahan bago mag-ADD o SELL."
        return onboarding_next(text)

    if "ADD:" in up:
        return add_reply(text)
    if "SELL:" in up:
        return sell_reply(text)
    return ("Seller, ito po ang mga utos: ADD: pangalan | bili | benta | stock | unit, "
            "SELL: pangalan | dami, 'AI ikaw na', o 'ako na'.")


# ==========================================================
# FASTAPI
# ==========================================================
class ChatReq(BaseModel):
    messages: List[Dict[str, Any]]
    max_tokens: Optional[int] = 300
    temperature: Optional[float] = 0.6
    story_block: Optional[int] = None


class BriefReq(BaseModel):
    brief: str


class MathReq(BaseModel):
    operation: str
    price: Optional[float] = None
    qty: Optional[float] = None
    subtotal: Optional[float] = None
    percent: Optional[float] = 0
    fixed: Optional[float] = 0
    cash_paid: Optional[float] = None
    total: Optional[float] = None
    gross_amount: Optional[float] = None
    inclusive: Optional[bool] = True
    sell_price: Optional[float] = None
    buy_price: Optional[float] = None
    net_profit: Optional[float] = None
    capital: Optional[float] = None
    monthly_expenses: Optional[float] = None
    avg_margin_pct: Optional[float] = None


def _resp(text, story_n=1):
    return {
        "object": "chat.completion",
        "model": MODEL_NAME,
        "story_block": story_n,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}}],
    }


app = FastAPI(title=f"{AGENT_NAME} Sales Worker")

# The page is served from a different origin than the tunnel, so the browser
# needs explicit CORS permission before it can POST /tts or read /health.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_NAME,
        "agent": AGENT_NAME,
        "convai_character": CONVAI_CHARACTER_ID,
        "story_blocks": len(STORY_BLOCKS),
    }


@app.get("/story")
def story():
    return {"blocks": STORY_BLOCKS, "convai_character": CONVAI_CHARACTER_ID}


@app.post("/v1/chat/completions")
def chat_completions(req: ChatReq, x_seller_token: Optional[str] = Header(default=None)):
    last_user = next(
        (m.get("content", "") for m in reversed(req.messages) if m.get("role") == "user"),
        "",
    )
    if not isinstance(last_user, str):
        last_user = str(last_user)
    last_user = last_user.strip()

    story_n = req.story_block or 1
    if story_n not in STORY_BY_N:
        story_n = 1

    with STATE_LOCK:
        if is_seller(x_seller_token):
            text = seller_reply(last_user)
        else:
            text = buyer_reply(
                last_user, req.max_tokens or 300, req.temperature or 0.6, story_n
            )
    return _resp(text, story_n)


@app.post("/generate_sales_pitch")
def generate_sales_pitch(req: BriefReq, x_seller_token: Optional[str] = Header(default=None)):
    if not is_seller(x_seller_token):
        raise HTTPException(status_code=403, detail="Seller lang.")
    with STATE_LOCK:
        text = llm_reply(
            f"Gumawa ng maikling sales pitch (3-4 pangungusap) para sa: {req.brief}. "
            "Natural na Tagalog, tapat, walang fake scarcity.",
            buyer_context(),
            max_tokens=250,
            temperature=0.7,
        )
    return {"pitch": text}


@app.post("/math")
def math_endpoint(req: MathReq):
    try:
        op = req.operation.lower()
        if op == "total":
            t, br = MathLogic.compute_total(req.price, req.qty)
            return {"ok": True, "result": t, "breakdown": br}
        if op == "discount":
            f, br = MathLogic.compute_discount(req.subtotal, req.percent or 0, req.fixed or 0)
            return {"ok": True, "result": f, "breakdown": br}
        if op == "change":
            return {"ok": True, "result": MathLogic.compute_change(req.cash_paid, req.total)}
        if op == "vat":
            return {"ok": True, "result": MathLogic.compute_vat(req.gross_amount, req.inclusive)}
        if op == "margin":
            return {"ok": True, "result": MathLogic.compute_profit_margin(req.sell_price, req.buy_price)}
        if op == "roi":
            return {"ok": True, "result": MathLogic.compute_roi(req.net_profit, req.capital)}
        if op == "bep":
            return {"ok": True, "result": MathLogic.compute_break_even(req.monthly_expenses, req.avg_margin_pct)}
        return {"ok": False, "error": f"Hindi kilalang operation: {op}"}
    except (ValueError, TypeError, ZeroDivisionError) as e:
        return {"ok": False, "error": str(e)}


class TTSReq(BaseModel):
    text: str
    voice_id: Optional[str] = None


@app.post("/tts")
async def tts(req: TTSReq):
    """Speak with edge-tts, free and keyless.

    Fish Audio was dropped: it bills per character and sends no CORS headers.
    edge-tts needs no key at all, so nothing secret ever touches this route.
    """
    text = (req.text or "").strip()[:600]
    if not text:
        raise HTTPException(status_code=400, detail="Walang text na isasalin sa boses.")
    voice = req.voice_id or TTS_VOICE
    buf = io.BytesIO()
    try:
        comm = edge_tts.Communicate(text, voice)
        async for chunk in comm.stream():
            if chunk["type"] == "audio":
                buf.write(chunk["data"])
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"edge-tts failed: {e}")
    audio = buf.getvalue()
    if not audio:
        raise HTTPException(status_code=502, detail="Walang audio na naibalik.")
    return Response(content=audio, media_type="audio/mpeg")


@app.post("/reset")
def reset_endpoint(x_seller_token: Optional[str] = Header(default=None)):
    if not is_seller(x_seller_token):
        raise HTTPException(status_code=403, detail="Seller lang.")
    with STATE_LOCK:
        reset_tindahan()
    return {"ok": True}


print("✅ Cell 3 done. API ready (hindi pa tumatakbo).")

⏳ Nilo-load ang SeaLLM... (1-3 minuto)
✅ SeaLLM loaded.
✅ Seller token loaded.
✅ Cell 3 done. API ready (hindi pa tumatakbo).


In [5]:
# ============================================================
# CELL 4 — Simulan ang server at Cloudflare tunnel
# ============================================================
import os
import re
import json
import time
import threading
import subprocess
import urllib.request
import uvicorn

PORT = 8000

def is_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        return True
    except Exception:
        return False

# 1) Simulan ang server sa background
if not is_up():
    server = uvicorn.Server(
        uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning")
    )
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(30):
        if is_up():
            break
        time.sleep(1)
print("Server:", "✅ buhay" if is_up() else "❌ hindi nag-start")

# 2) I-download ang cloudflared kung wala pa
CF = "./cloudflared"
if not os.path.exists(CF):
    subprocess.run([
        "wget", "-q", "-O", CF,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ], check=True)
    os.chmod(CF, 0o755)

# 3) Simulan ang tunnel
LOG = "./tunnel.log"
tunnel = subprocess.Popen(
    [CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(2)
    with open(LOG) as f:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
    if m:
        public_url = m.group(0)
        break

print("Public URL:", public_url or "Hindi nakuha. Tingnan ang ./tunnel.log")
if public_url:
    print("Endpoint para sa workers:", f"{public_url}/v1/chat/completions")

Server: ✅ buhay
Public URL: https://ballet-departments-height-partly.trycloudflare.com
Endpoint para sa workers: https://ballet-departments-height-partly.trycloudflare.com/v1/chat/completions


In [6]:
# ============================================================
# CELL 5 — Reset, setup, at test
# ============================================================
import os, json, urllib.request

PORT = 8000


def _read_env_file(path="./.env"):
    """Standalone: SELLER_TOKEN comes from .env, never from Kaggle Secrets."""
    values = {}
    if not os.path.exists(path):
        return values
    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            values[key.strip()] = val.strip().strip('"').strip("'")
    return values


TOKEN = os.environ.get("SELLER_TOKEN", "").strip() or _read_env_file().get("SELLER_TOKEN", "").strip()
if not TOKEN:
    print("⚠️ Walang SELLER_TOKEN sa .env — naka-disable ang seller commands.")


def _post(path, payload, seller=False):
    headers = {"Content-Type": "application/json"}
    if seller:
        headers["X-Seller-Token"] = TOKEN
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.loads(r.read())


def call(msg, seller=False):
    res = _post("/v1/chat/completions", {"messages": [{"role": "user", "content": msg}]}, seller)
    return res["choices"][0]["message"]["content"]


# 0) Reset ng lumang data
print("RESET:", _post("/reset", {}, seller=True))

# 1) Setup ng seller: "mulai" muna, saka ang mga sagot
setup_steps = [
    "mulai",
    "SariSwerte",
    "Inday",
    "FB Live",
    "5000",
    "1000",
    "50",
    "500",
    "7-day return kung may sira",
]
for step in setup_steps:
    print(f"SETUP [{step}] ->", call(step, seller=True))

# 2) Produkto at benta
print("ADD ->", call("ADD: Coke 1.5L | 65 | 75 | 20 | bote", seller=True))
print("SELL ->", call("SELL: Coke 1.5L | 2", seller=True))

# 3) Test ng buyer at take-over
print("1) Buyer, naka-off ->", call("Magkano ang Coke?"))
print("2) Seller, AI on ->", call("AI ikaw na", seller=True))
print("3) Buyer, presyo ->", call("Magkano ang Coke?"))
print("4) Buyer, stock ->", call("May stock pa ba ang Coke?"))
print("5) Buyer, wala sa listahan ->", call("Magkano ang Pepsi?"))
print("6) Buyer, sensitibo ->", call("Sira ang binili ko, gusto ko ng refund"))
print("7) Seller, ibalik ->", call("ako na", seller=True))
print("8) Buyer, naka-off ulit ->", call("Magkano ang Coke?"))

✅ Na-reset. Fresh start!
RESET: {'ok': True}
SETUP [mulai] -> Boss, ano po ang pangalan ng tindahan/shop ninyo? (Hal. Nena's Store)
SETUP [Nena's Store] -> Ano po ang pangalan ninyo, boss?
SETUP [Nena] -> Saan po kayo nag-live selling? (FB Live, TikTok, Shopee, o 'walk-in')
SETUP [FB Live] -> Magkano po ang puhunan ninyo? (Hal. 5000)
SETUP [5000] -> Magkano po ang buwanang gastos ninyo? (Kuryente, tubig, internet, renta)
SETUP [1000] -> Magkano po ang standard shipping fee? (0 kung pickup)
SETUP [50] -> Magkano po ang minimum order para sa free shipping? (0 kung wala)
SETUP [500] -> Ano po ang return/refund policy ninyo? (Hal. '7-day return kung may sira')
SETUP [7-day return kung may sira] -> Tapos na ang setup!

SHOP: Nena's Store | SELLER: Nena | PLATFORM: FB Live
TAKE-OVER ACTIVE: False
Puhunan: ₱5000.00 | Buwanang gastos: ₱1000.00
Shipping: ₱50.00 | Free ship min: ₱500.00
Return policy: 7-day return kung may sira
Produkto: 0 | Inventory value: ₱0.00
Gross sales: ₱0.00 | COGS: ₱0.0